In [1]:
import straw
import numpy as np
import math
import os

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

os.chdir('../..')
from load import generate_coverage_profile_intra, load_GC_comtent, getmappability, getGCpercent, getFragmentsNumber, load_num_res_site
from binlib import output_bin
from metric import calculate_correlation_with_groundtruth, calculate_precision_racall
from convert import convert_bin_to_bw

# back to current filder
os.chdir('Exps/exp4_intra_chr')

ref_genome_folder = '../../Data/ref_genome/hg19'
contact_matrix = '../../Data/ContactMatrix/GSE63525_K562_combined.hic'
exclude_chr = ['chrM', 'chrY']
resolution = 50000


# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)
print('chr length : \n',chr_length)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1
print('chr bin number : \n', chr_bin_number)



chr length : 
 {'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895, 'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753, 'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520, 'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560}
chr bin number : 
 {'chr1': 4986, 'chr2': 4864, 'chr3': 3961, 'chr4': 3824, 'chr5': 3619, 'chr6': 3423, 'chr7': 3183, 'chr8': 2928, 'chr9': 2825, 'chr10': 2711, 'chr11': 2701, 'chr12': 2678, 'chr13': 2304, 'chr14': 2147, 'chr15': 2051, 'chr16': 1808, 'chr17': 1624, 'chr18': 1562, 'chr19': 1183, 'chr20': 1261, 'chr21': 963, 'chr22': 1027, 'chrX': 3106}


### load bias

In [8]:
# GC_dict = load_GC_comtent(ref_genome_folder + '/hg19_1kb_GCPercent.bed')

# mappability_dict = dict()
# for key in chr_length.keys() :
#     mappability_dict[key] = np.array( getmappability(ref_genome_folder + '/hg19_mappability_50mer.bdg.gz', key, 1, chr_length[key], resolution) )

# num_res_sites_dict = load_num_res_site(ref_genome_folder + '/hg19_HindIII_enzymeSites.txt')

In [2]:
def getRestrictionSitesInfo(restrictionSites):
	inf = open(restrictionSites)
	sitesInfo = {}
	for line in inf:
		line = line.strip().split()
		chrom = line[0]
		sites = line[1:]
		sitesInfo[chrom] = sites
	return sitesInfo

mappability_dict = dict()
GC_dict = dict()
num_res_sites_dict = dict()
for key in chr_length.keys() :
    mappability_dict[key] = np.array( getmappability(ref_genome_folder + '/hg19_mappability_50mer.bdg.gz', key, 1, chr_length[key], resolution) )
    GC_dict[key] = np.array( getGCpercent(ref_genome_folder + '/hg19_1kb_GCPercent.bed', key, resolution) )
    num_res_sites_dict[key] = np.array( getFragmentsNumber(getRestrictionSitesInfo(ref_genome_folder + '/hg19_HindIII_enzymeSites.txt'), key, resolution, chr_length[key]) )

In [3]:
coverage_profile_intra_dict = dict()
for key in chr_length.keys():
    print(f'Parsing contacts between {key} and other')
    coverage_profile_intra_dict[key] = generate_coverage_profile_intra(contact_matrix, trg_chr = key, resolution = resolution)

Parsing contacts between chr1 and other
Parsing contacts between chr2 and other
Parsing contacts between chr3 and other
Parsing contacts between chr4 and other
Parsing contacts between chr5 and other
Parsing contacts between chr6 and other
Parsing contacts between chr7 and other
Parsing contacts between chr8 and other
Parsing contacts between chr9 and other
Parsing contacts between chr10 and other
Parsing contacts between chr11 and other
Parsing contacts between chr12 and other
Parsing contacts between chr13 and other
Parsing contacts between chr14 and other
Parsing contacts between chr15 and other
Parsing contacts between chr16 and other
Parsing contacts between chr17 and other
Parsing contacts between chr18 and other
Parsing contacts between chr19 and other
Parsing contacts between chr20 and other
Parsing contacts between chr21 and other
Parsing contacts between chr22 and other
Parsing contacts between chrX and other


In [3]:
# write to file
f = open('bias_with_intra_rowsum_allchroms_nonzeros.txt','w+')
f.write(f'chrom\tbin_num\tcutSitesNumber\tgcper\tmapscore\trowsum\n')
for key in chr_bin_number.keys():
    for i in range(chr_bin_number[key]):
        num_res_sites = num_res_sites_dict[key][i]
        mappability = mappability_dict[key][i]
        GC = GC_dict[key][i]
        rowsum = coverage_profile_intra_dict[key][i]
        if num_res_sites == mappability == GC ==0 :
            continue
        f.write(f'{key}\t{i}\t{num_res_sites}\t{GC}\t{mappability}\t{rowsum}\n')

### Run script 

In [5]:
os.system('Rscript ../../GAM.R --input bias_with_intra_rowsum_allchroms_nonzeros.txt --log GAM_log --output residual.txt')

Loading required package: nlme
This is mgcv 1.8-38. For overview type 'help("mgcv-package")'.


[1] "Loading input data ... "
[1] "Fitting GAM model ... "
[1] "Calculating residuals ..."
null device 
          1 
[1] "Done!"


0

### Merge residual to orig file and output to binfile

In [6]:
bias_rowsum = open('bias_with_intra_rowsum_allchroms_nonzeros.txt','r')
residual = open('residual.txt','r')

res_dict = dict()
for key in chr_bin_number.keys():
    res_dict[key] = np.zeros(chr_bin_number[key])

bias_rowsum_lines = bias_rowsum.readlines()[1:]
residual_line = residual.readlines()

for i, bias_line in enumerate(bias_rowsum_lines):
    bias_line_split = bias_line.strip().split()
    chr_ = bias_line_split[0]
    bin_num = int(bias_line_split[1])
    res = residual_line[i]
    res_dict[chr_][bin_num] = float(res)

In [7]:
output_bin(res_dict, trg_output_dir = 'exp4_bin')

In [8]:
# output for visualization
convert_bin_to_bw('exp4_bin', './exp4.bw')

### Run BICseq-seq manually 

(linux)
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp4_bin/_config.txt exp4_bin/exp4_lambda3.seg --title intra_HiC_K562_lambda3 --fig exp4_bin/exp4_lambda3.png --lambda 3 --bootstrap

### metric

In [16]:
lambda_ = [1,2,3]

In [18]:
for l in lambda_ : 
    print(f"lambda {l}")
    calculate_correlation_with_groundtruth(f'exp4_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
    print('\n')

lambda 1
Pearson r :  0.897
Spearman r :  0.806


lambda 2
Pearson r :  0.899
Spearman r :  0.806


lambda 3
Pearson r :  0.726
Spearman r :  0.792




In [19]:
for l in lambda_ :
    print(f"lambda {l}")
    calculate_precision_racall(f'exp4_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
    print('\n')

lambda 1
Number of ground truth : 74
Number of target : 48
Number of hits : 35
precision : 0.729
recall : 0.473
average hit fraction : 0.875


lambda 2
Number of ground truth : 74
Number of target : 28
Number of hits : 24
precision : 0.857
recall : 0.324
average hit fraction : 0.875


lambda 3
Number of ground truth : 74
Number of target : 25
Number of hits : 20
precision : 0.8
recall : 0.27
average hit fraction : 0.833


